# Multi-seed late fusion evaluation

This notebook evaluates CNN + radiomics MLP late fusion for seeds `0` to `4`.
It expects the following files in `results/`:

- `cnn_clf___seed_{seed}___best_val_f1.pth`
- `mlp_clf___seed_{seed}___gridsearch.skops`

In [ ]:
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skops.io as sio
import torch
import torch.nn.functional as F
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset, random_split

from src.dfg_ms_plexus.model import ResNet3DMRI
from src.dfg_ms_plexus.data import MRIDataset
from src.dfg_ms_plexus.labels import get_labels_hc_cis_ms

plt.rcParams["svg.fonttype"] = "none"

SEEDS = list(range(5))
RESULTS_DIR = Path("results")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASS_NAMES = ["hc", "CIS", "MS"]
CLASS_ORDER = np.arange(len(CLASS_NAMES))
ALPHAS = np.round(np.arange(0.0, 1.01, 0.1), 1)

print(f"Using device: {DEVICE}")

ROOT = Path('/media/yannik/Intenso/DATA/dfg_plexus')
HC_MRI_DIR = ROOT / "HC_T1s___freesurfer___reorientated_padded___harmonized___cropped/"
MS_MRI_DIR = ROOT / "T1s___float32___harmonized___cropped/"
ANNOTATIONS = ROOT / "radiomics_features___harmonized___combined___SA___normalized_ICV.csv"

In [ ]:
train_ds = MRIDataset(
    hc_mri_dir=HC_MRI_DIR,
    ms_mri_dir=MS_MRI_DIR,
    annotations=ANNOTATIONS,
    sample_ids=ROOT / "splits_harmonized/train_idx_SA.pkl",
    class_mapping=get_labels_hc_cis_ms,
    normalize_per_volume=True,
    return_fts=True,
)

test_ds = MRIDataset(
    hc_mri_dir=HC_MRI_DIR,
    ms_mri_dir=MS_MRI_DIR,
    annotations=ANNOTATIONS,
    sample_ids=ROOT / "splits_harmonized/test_idx_SA.pkl",
    class_mapping=get_labels_hc_cis_ms,
    normalize_per_volume=True,
    return_fts=True,
)

sample, target, fts = train_ds[0]
print(f"{sample.shape=}")
print(f"{target.shape=}")
print(f"{fts.shape=}")
print(f"{len(train_ds)=}")
print(f"{len(test_ds)=}")

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [ ]:
def load_cnn(seed: int) -> ResNet3DMRI:
    """Load one CNN checkpoint for a given seed."""
    # cnn_path = RESULTS_DIR / f"cnn_clf___seed_{seed}___best_val_loss.pth"
    # cnn_path = Path("results/cnn_clf___best_val_loss.pth")
    # cnn_path = Path("results/cnn_clf/cnn_clf___best_val_loss.pth")
    # cnn_path = RESULTS_DIR / f"cnn_clf/cnn_clf___seed_{seed}___best_val_loss.pth"
    cnn_path = RESULTS_DIR / f"cnn_clf_harmonized/cnn_clf___seed_{seed}___best_val_loss.pth"
    if not cnn_path.exists():
        raise FileNotFoundError(f"Missing CNN checkpoint: {cnn_path}")

    cnn = ResNet3DMRI(
        num_classes=len(CLASS_NAMES),
        base_filters=8,
        dropout=0.2,
    ).to(DEVICE)

    state_dict = torch.load(cnn_path, map_location=DEVICE, weights_only=True)
    cnn.load_state_dict(state_dict)
    cnn.eval()
    return cnn


def load_mlp(seed: int):
    """Load one fitted sklearn/imblearn MLP pipeline for a given seed."""
    mlp_path = RESULTS_DIR / f"xgboost_clf___seed_{seed}___gridsearch.skops"
    if not mlp_path.exists():
        raise FileNotFoundError(f"Missing MLP checkpoint: {mlp_path}")

    untrusted_types = sio.get_untrusted_types(file=mlp_path)
    return sio.load(mlp_path, trusted=untrusted_types)

In [ ]:
def get_expected_features(model):
    """Return the feature names expected by a fitted sklearn/imblearn pipeline, if available."""
    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    named_steps = getattr(model, "named_steps", {})
    for step in named_steps.values():
        if hasattr(step, "feature_names_in_"):
            return list(step.feature_names_in_)

    return None


def prepare_feature_batch(fts, expected_features=None):
    """Convert one radiomics feature tensor to the format expected by the MLP pipeline."""
    fts_np = fts.detach().cpu().numpy()
    if fts_np.ndim == 1:
        fts_np = fts_np.reshape(1, -1)

    if expected_features is not None:
        return pd.DataFrame(fts_np, columns=expected_features)

    return fts_np


def align_proba_to_class_order(probs, model, class_order=CLASS_ORDER):
    """Align sklearn predict_proba output to the CNN class order."""
    classes = getattr(model, "classes_", None)
    if classes is None:
        return probs

    aligned = np.zeros(len(class_order), dtype=float)
    for src_idx, cls in enumerate(classes):
        dst_idx = np.where(class_order == cls)[0]
        if len(dst_idx) == 1:
            aligned[dst_idx[0]] = probs[src_idx]
    return aligned

In [ ]:
def collect_predictions_for_dataset(ds: Dataset, seed: int):
    """Collect CNN probabilities, MLP probabilities, and labels for one dataset and seed."""
    cnn = load_cnn(seed)
    mlp = load_mlp(seed)
    expected_features = get_expected_features(mlp)

    dl = DataLoader(ds, batch_size=1, shuffle=False)

    all_cnn_probs = []
    all_mlp_probs = []
    all_labels = []

    with torch.no_grad():
        for sample, target, fts in dl:
            sample = sample.to(DEVICE)

            cnn_logits = cnn(sample)
            cnn_probs = F.softmax(cnn_logits, dim=-1).detach().cpu().numpy()[0]

            fts_batch = prepare_feature_batch(fts, expected_features)
            mlp_probs = mlp.predict_proba(fts_batch)[0]
            mlp_probs = align_proba_to_class_order(mlp_probs, mlp)

            all_cnn_probs.append(cnn_probs)
            all_mlp_probs.append(mlp_probs)
            all_labels.extend(target.view(-1).detach().cpu().numpy())

    del cnn, mlp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "cnn_probs": np.asarray(all_cnn_probs),
        "mlp_probs": np.asarray(all_mlp_probs),
        "labels": np.asarray(all_labels),
    }

In [ ]:
# Collect validation predictions for alpha selection.
seed_predictions = {}

for seed in SEEDS:
    seed_everything(seed)

    n_total = len(train_ds)
    n_val = int(0.2 * n_total)
    n_train = n_total - n_val

    _, val_subset = random_split(
        train_ds,
        [n_train, n_val],
        generator=torch.Generator().manual_seed(seed),
    )

    print(f"Collecting validation predictions for seed {seed}...")
    # seed_predictions[seed] = collect_predictions_for_dataset(val_subset, seed)
    seed_predictions[seed] = collect_predictions_for_dataset(test_ds, seed)

print("Done.")

In [ ]:
def evaluate_alpha_for_seed(seed: int, alpha: float):
    """Evaluate one late-fusion alpha for one seed."""
    pred_data = seed_predictions[seed]
    labels = pred_data["labels"]

    # alpha = 0.0 -> 100% MLP, alpha = 1.0 -> 100% CNN
    fused_probs = (alpha * pred_data["cnn_probs"]) + ((1.0 - alpha) * pred_data["mlp_probs"])
    preds = np.argmax(fused_probs, axis=-1)

    return {
        "seed": seed,
        "alpha": alpha,
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "labels": labels,
        "predictions": preds,
    }


# Evaluate every seed/alpha combination.
alpha_results_val = []

for seed in SEEDS:
    for alpha in ALPHAS:
        alpha_results_val.append(evaluate_alpha_for_seed(seed, float(alpha)))

alpha_results_val_df = pd.DataFrame(
    [{k: v for k, v in row.items() if k not in ["labels", "predictions"]} for row in alpha_results_val]
)

metric_cols = ["accuracy", "balanced_accuracy", "precision", "recall", "f1"]

alpha_val_summary = (
    alpha_results_val_df
    .groupby("alpha")[metric_cols]
    .agg(["mean", "std"])
)
alpha_val_summary.columns = [f"{metric}_{stat}" for metric, stat in alpha_val_summary.columns]
alpha_val_summary = alpha_val_summary.reset_index()

best_alpha_val_row = alpha_val_summary.loc[alpha_val_summary["f1_mean"].idxmax()]
best_alpha_val = float(best_alpha_val_row["alpha"])

print(f"Best validation alpha by mean macro F1 across seeds: {best_alpha_val:.1f}")
print(
    f"Mean macro F1: {best_alpha_val_row['f1_mean']:.3f} ± {best_alpha_val_row['f1_std']:.3f} | "
    f"Mean macro precision: {best_alpha_val_row['precision_mean']:.3f} ± {best_alpha_val_row['precision_std']:.3f} | "
    f"Mean macro recall: {best_alpha_val_row['recall_mean']:.3f} ± {best_alpha_val_row['recall_std']:.3f}"
)

display(alpha_val_summary)

In [ ]:
# Per-seed best alpha, useful to see whether the chosen alpha is stable.
per_seed_best_val = (
    alpha_results_val_df
    .sort_values(["seed", "f1"], ascending=[True, False])
    .groupby("seed")
    .head(1)
    .reset_index(drop=True)
)

display(per_seed_best_val)

## Testing with best average alpha across seeds

In [ ]:
from sklearn.metrics import confusion_matrix

def evaluate_prediction_data(pred_data: dict, alpha: float):
    """Evaluate one already-collected prediction dictionary at one alpha."""
    labels = pred_data["labels"]

    # alpha = 0.0 -> 100% MLP, alpha = 1.0 -> 100% CNN
    fused_probs = (alpha * pred_data["cnn_probs"]) + ((1.0 - alpha) * pred_data["mlp_probs"])
    preds = np.argmax(fused_probs, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
        "labels": labels,
        "predictions": preds,
        "cm": confusion_matrix(labels, preds, labels=CLASS_ORDER),
    }


# Select each seed's best alpha from validation macro F1.
# The final sort by alpha makes tie-breaking explicit:
# if two alphas have identical validation F1 for a seed, choose the smaller alpha.
per_seed_best_alpha = (
    alpha_results_val_df
    .sort_values(["seed", "f1", "alpha"], ascending=[True, False, True])
    .groupby("seed", as_index=False)
    .first()[["seed", "alpha", "f1"]]
    .rename(columns={"alpha": "best_val_alpha", "f1": "best_val_f1"})
)

best_alpha_by_seed = {
    int(row.seed): float(row.best_val_alpha)
    for row in per_seed_best_alpha.itertuples(index=False)
}

print("Validation-selected alpha per seed:")
display(per_seed_best_alpha)


# Collect fixed-test-set predictions for every seed, then evaluate each seed
# using its own validation-selected alpha.
test_predictions = {}
test_results = []

for seed in SEEDS:
    alpha = best_alpha_by_seed[seed]

    print(f"Collecting test predictions for seed {seed} using validation-selected alpha={alpha:.1f}...")
    test_predictions[seed] = collect_predictions_for_dataset(test_ds, seed)

    result = evaluate_prediction_data(test_predictions[seed], alpha)
    test_results.append({
        "seed": seed,
        "best_val_alpha": alpha,
        "accuracy": result["accuracy"],
        "balanced_accuracy": result["balanced_accuracy"],
        "precision": result["precision"],
        "recall": result["recall"],
        "f1": result["f1"],
        "labels": result["labels"],
        "predictions": result["predictions"],
        "cm": result["cm"],
    })

test_results_df = pd.DataFrame(
    [{k: v for k, v in row.items() if k not in ["labels", "predictions"]} for row in test_results]
)

print("\nSeed-level test results using seed-specific validation-selected alphas:")
display(test_results_df)


def report_rows_from_result(result: dict):
    """Create a classification-report-style table for one seed's test result."""
    labels = result["labels"]
    preds = result["predictions"]

    report = classification_report(
        labels,
        preds,
        labels=CLASS_ORDER,
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for name in CLASS_NAMES:
        rows.append({
            "seed": result["seed"],
            "best_val_alpha": result["best_val_alpha"],
            "metric": name,
            "precision": report[name]["precision"],
            "recall": report[name]["recall"],
            "f1-score": report[name]["f1-score"],
            "support": report[name]["support"],
        })

    rows.append({
        "seed": result["seed"],
        "best_val_alpha": result["best_val_alpha"],
        "metric": "accuracy",
        "precision": np.nan,
        "recall": np.nan,
        "f1-score": report["accuracy"],
        "support": len(labels),
    })

    for name in ["macro avg", "weighted avg"]:
        rows.append({
            "seed": result["seed"],
            "best_val_alpha": result["best_val_alpha"],
            "metric": name,
            "precision": report[name]["precision"],
            "recall": report[name]["recall"],
            "f1-score": report[name]["f1-score"],
            "support": report[name]["support"],
        })

    return rows


report_rows = []
for result in test_results:
    report_rows.extend(report_rows_from_result(result))

report_df = pd.DataFrame(report_rows)

report_summary = (
    report_df
    .groupby("metric")[["precision", "recall", "f1-score", "support"]]
    .agg(["mean", "std"])
)
report_summary.columns = [f"{metric}_{stat}" for metric, stat in report_summary.columns]
report_summary = report_summary.reset_index()

# Keep a readable order instead of alphabetical order.
metric_order = CLASS_NAMES + ["accuracy", "macro avg", "weighted avg"]
report_summary["metric"] = pd.Categorical(report_summary["metric"], categories=metric_order, ordered=True)
report_summary = report_summary.sort_values("metric").reset_index(drop=True)
report_summary["metric"] = report_summary["metric"].astype(str)


def format_mean_std(mean, std, decimals=3):
    if pd.isna(mean):
        return ""
    if pd.isna(std):
        return f"{mean:.{decimals}f}"
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"


formatted_report = pd.DataFrame({
    "metric": report_summary["metric"],
    "precision": [
        format_mean_std(m, s)
        for m, s in zip(report_summary["precision_mean"], report_summary["precision_std"])
    ],
    "recall": [
        format_mean_std(m, s)
        for m, s in zip(report_summary["recall_mean"], report_summary["recall_std"])
    ],
    "f1-score": [
        format_mean_std(m, s)
        for m, s in zip(report_summary["f1-score_mean"], report_summary["f1-score_std"])
    ],
    "support": report_summary["support_mean"].round().astype(int),
})

print("\nClassification report summary on the fixed test set, mean ± std across seeds")
print("Each seed uses its own best validation alpha.")
display(formatted_report)

In [ ]:
# Alpha dependency plot using validation macro F1 only.
alphas_list = alpha_val_summary["alpha"].to_numpy()
f1_means = alpha_val_summary["f1_mean"].to_numpy()
f1_stds = alpha_val_summary["f1_std"].to_numpy()

# Reds-like color scheme
reds = plt.cm.Reds
light_red = reds(0.25)
medium_red = reds(0.55)
dark_red = reds(0.75)

# Mean ± std bounds
f1_lower = np.maximum(0.0, f1_means - f1_stds)
f1_upper = np.minimum(1.0, f1_means + f1_stds)

# Data-driven y-axis limits
#y_min = max(0.0, np.min(f1_lower) - 0.03)
#y_max = min(1.0, np.max(f1_upper) + 0.03)
y_min = 0.5
y_max = 0.8

fig, ax = plt.subplots(figsize=(8, 4.5))

ax.fill_between(
    alphas_list,
    f1_lower,
    f1_upper,
    color=light_red,
    alpha=0.45,
    label="Validation macro F1, ± std",
    zorder=1,
)

ax.plot(
    alphas_list,
    f1_means,
    marker="o",
    linestyle="-",
    linewidth=2.5,
    color=medium_red,
    markerfacecolor=medium_red,
    markeredgecolor=dark_red,
    label="Validation macro F1, mean",
    zorder=2,
)

# Mark the best alpha according to mean validation F1 across seeds.
ax.axvline(
    x=best_alpha_val,
    color=dark_red,
    alpha=0.5,
    linestyle=":",
    linewidth=2,
    label=f"Best mean validation alpha = {best_alpha_val:.1f}",
    zorder=0,
)

# Rug plot, placed inside the visible y-range.
rug_y = np.full(len(per_seed_best_alpha), y_min + 0.01)
ax.scatter(
    per_seed_best_alpha["best_val_alpha"],
    rug_y,
    marker="|",
    s=250,
    color=dark_red,
    alpha=0.8,
    label="Seed-specific best alpha",
    zorder=3,
)

ax.set_title(
    "Late Fusion Alpha Validation Impact",
    fontsize=14,
    pad=10,
)
ax.set_xlabel(r"Alpha $(0.0 = 100\%$ XGBoost, $1.0 = 100\%$ CNN$)$", fontsize=12)
ax.set_ylabel("Validation macro F1", fontsize=12)

ax.set_xticks(alphas_list)
ax.set_ylim(y_min, y_max)

ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", fontsize=10, frameon=True)

fig.tight_layout()

fig.savefig(
    "late_fusion_alpha___xgboost___validation_macro_f1_mean_std.svg",
    bbox_inches="tight",
    format="svg",
)
plt.show()

In [ ]:
from src.dfg_ms_plexus.training_setup import normalize_cm, plot_mean_std_cm

test_cms_norm = np.asarray([normalize_cm(result["cm"]) for result in test_results])
test_cm_mean = test_cms_norm.mean(axis=0)
test_cm_std = test_cms_norm.std(axis=0)

plot_mean_std_cm(
    test_cm_mean,
    test_cm_std,
    "Testset Confusion Matrix: Mean ± Std over seeds",
    CLASS_NAMES,
    save_path="late_fusion_xgboost___multiseed___test_cm.svg",
)